# Lane 2: Diff-JEPA - Google Colab Runner
**Project:** Action-Asymmetric Differential Isotropic JEPA  
**Author:** Jaime Jarauta  
**Pre-Registered Question:** Does subtracting an unconditioned passive background head from an action-conditioned causal head with SIGReg isolate causal affordances under heavy visual clutter?

Configured for Google Colab (Free T4 or Pro) and local GPUs.


In [ ]:
# Step 1: Pre-flight Hardware Verification
import sys, os, time, math
import torch

print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    device = torch.device("cuda")
    dev_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Device: {dev_name} | VRAM: {vram_gb:.2f} GB")
    torch.set_float32_matmul_precision("high")
    torch.backends.cudnn.benchmark = True
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Device: Apple Silicon MPS")
else:
    device = torch.device("cpu")
    print("Device: CPU")


In [ ]:
# Step 2: Dependencies & Imports
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q uv
    !uv pip install --system -q pyyaml wandb pytest

if "src" not in sys.path:
    sys.path.insert(0, "src")

from diff_jepa.models import ActionAsymmetricDualPredictor, VisualStateEncoder
from diff_jepa.sigreg import SIGRegLoss
from diff_jepa.metrics import compute_effective_rank

print("Diff-JEPA modules loaded successfully.")


In [ ]:
# Step 3: Run Push-T Affordance Smoke Test under 80% Clutter
import torch.nn.functional as F

torch.manual_seed(2026)
d_causal, d_action, d_noise = 16, 4, 112
d_total = d_causal + d_noise
batch_size, n_steps = 64, 300

encoder = VisualStateEncoder(d_raw=d_total, d_state=d_total).to(device)
predictor = ActionAsymmetricDualPredictor(d_state=d_total, d_action=d_action, d_out=d_total).to(device)
sigreg_diff = SIGRegLoss(d=d_total, n_slices=32).to(device)
sigreg_enc = SIGRegLoss(d=d_total, n_slices=32).to(device)

opt = torch.optim.Adam(list(encoder.parameters()) + list(predictor.parameters()), lr=1e-3)

t0 = time.time()
losses = []
latents = []
W_action = torch.randn(d_action, d_causal, device=device) * 0.5

for step in range(n_steps):
    s_causal = torch.randn(batch_size, d_causal, device=device)
    action = torch.randn(batch_size, d_action, device=device)
    s_next_causal = 1.2 * s_causal + action @ W_action
    
    bg_shared = 3.0 * torch.randn(batch_size, d_noise, device=device)
    raw_curr = torch.cat([s_causal, bg_shared + 0.2*torch.randn(batch_size, d_noise, device=device)], dim=-1)
    raw_next = torch.cat([s_next_causal, bg_shared + 0.2*torch.randn(batch_size, d_noise, device=device)], dim=-1)
    
    s_curr = encoder(raw_curr)
    with torch.no_grad():
        s_target = encoder(raw_next)
        
    pred_diff = predictor(s_curr, action)
    target_norm = (s_target - s_target.mean(dim=0, keepdim=True)) / (s_target.std(dim=0, keepdim=True) + 1e-5)
    
    l_pred = F.mse_loss(pred_diff, target_norm)
    l_sigreg_diff = sigreg_diff(pred_diff)
    l_sigreg_enc = sigreg_enc(s_curr)
    loss = l_pred + 0.1 * l_sigreg_diff + 0.05 * l_sigreg_enc
    
    opt.zero_grad()
    loss.backward()
    opt.step()
    
    losses.append(loss.item())
    if step >= n_steps - 50:
        latents.append(pred_diff.detach())
    if step % 50 == 0 or step == n_steps - 1:
        print(f"Step {step:03d} | Loss: {loss.item():.4f} | Lambda: {predictor.get_lambda().item():.3f}")

elapsed = time.time() - t0
final_latents = torch.cat(latents, dim=0)
erank = compute_effective_rank(final_latents)
erank_ratio = erank / d_total
print(f"\nCompleted in {elapsed:.2f}s | Effective Rank: {erank:.2f} / {d_total} (Ratio: {erank_ratio:.2f})")


In [ ]:
# Step 4: Pre-Registered Gate Assessment
passed = erank_ratio >= 0.70 and losses[-1] < losses[0]
print("=" * 60)
print(f"Effective Rank Ratio: {erank_ratio:.2f} (Target: >= 0.70)")
print(f"Initial Loss: {losses[0]:.4f} -> Final Loss: {losses[-1]:.4f}")
print(f"Pre-Registered Gate: {'PASSED (No collapse & loss decreased)' if passed else 'FAILED (Kill)'}")
print("=" * 60)
